# Pipeline de Treinamento — Previsão de `geek_rating` (BoardGameGeek)

Este notebook treina e compara 4 modelos para prever o **geek_rating** (Bayesian rating do BGG) a partir da base já normalizada em `normalization.ipynb`:

| Modelo | Tipo | Target usado |
|---|---|---|
| Regressão Linear | Regressão | `geek_rating` contínuo |
| Random Forest Regressor | Regressão | `geek_rating` contínuo |
| Gradient Boosting Regressor | Regressão | `geek_rating` contínuo |
| Regressão Logística | Classificação (10 classes) | `geek_rating` discretizado em 10 faixas |

**Por que a Regressão Logística é tratada separadamente?**
Regressão logística é, por natureza, um modelo de **classificação**, não de regressão contínua. Para usá-la na previsão de `geek_rating`, discretizamos o valor contínuo em **10 faixas (bins) de largura igual**, cobrindo o intervalo observado de `geek_rating` no conjunto de treino — ou seja, transformamos o problema em uma classificação multiclasse com classes de 0 a 9 (decis de amplitude, não de frequência).

> ⚠️ Os limites das faixas (bins) são calculados **apenas com os dados de treino** e depois aplicados ao teste, para evitar vazamento de informação (data leakage).

Todos os 4 modelos têm seus hiperparâmetros otimizados via `GridSearchCV`.

## 1. Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42


## 2. Carregamento dos dados

Carregamos `bgg_normalized.csv`, gerado pelo notebook `normalization.ipynb`. A base já está limpa, sem valores nulos, com variáveis categóricas codificadas (one-hot para categorias/mecânicas), publisher e descrição vetorizados, e features numéricas padronizadas com `RobustScaler`. O `geek_rating` (nosso target) permanece na escala original.

In [ ]:
df = pd.read_csv("bgg_normalized.csv")

print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Valores nulos totais: {df.isna().sum().sum()}")
df.head()


In [ ]:
df["geek_rating"].describe()


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["geek_rating"], bins=50, kde=True)
plt.title("Distribuição de geek_rating")
plt.xlabel("geek_rating")
plt.ylabel("Frequência")
plt.show()


**Observação importante sobre a distribuição:** o `geek_rating` é fortemente assimétrico à direita. A maioria dos jogos se concentra perto do "piso" do Bayesian average do BGG (em torno de 5.5), e poucos jogos atingem ratings altos (>7.5). Isso afeta diretamente a versão de classificação (Regressão Logística): as faixas mais altas terão poucas amostras, o que é esperado e será discutido nos resultados.

## 3. Separação de features e targets

- `X`: todas as colunas exceto `geek_rating`.
- `y_reg`: o `geek_rating` contínuo (usado em Regressão Linear, Random Forest e Gradient Boosting).
- O target de classificação (`y_clf`) será derivado de `y_reg` *depois* do split treino/teste, para não vazar informação do teste na definição das faixas.

In [ ]:
X = df.drop(columns=["geek_rating"])
y_reg = df["geek_rating"]

print(f"X: {X.shape}")
print(f"y_reg: {y_reg.shape}")


## 4. Split treino/teste

Usamos `train_test_split` com 80% treino / 20% teste. Como as features já vêm padronizadas (RobustScaler) e codificadas do notebook de normalização, não há necessidade de pré-processamento adicional aqui.

In [ ]:
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Treino: {X_train.shape[0]:,} amostras")
print(f"Teste:  {X_test.shape[0]:,} amostras")


## 5. Discretização do target para a Regressão Logística

Criamos 10 faixas de **largura igual** (bins lineares, `np.linspace`) cobrindo o intervalo `[min, max]` de `geek_rating` observado **no treino**. As extremidades dos bins são abertas para `-inf`/`+inf`, garantindo que qualquer valor do teste (mesmo fora do range do treino) seja classificado em alguma faixa, sem gerar `NaN`.

As classes resultantes vão de `0` (faixa mais baixa) a `9` (faixa mais alta).

In [ ]:
N_BINS = 10

bin_edges = np.linspace(y_train_reg.min(), y_train_reg.max(), N_BINS + 1)

# Rótulos legíveis para cada faixa (antes de abrir as extremidades)
bin_labels_desc = [f"{bin_edges[i]:.3f} – {bin_edges[i+1]:.3f}" for i in range(N_BINS)]

# Abre as extremidades para +-inf, para cobrir qualquer valor do teste
bin_edges_open = bin_edges.copy()
bin_edges_open[0] = -np.inf
bin_edges_open[-1] = np.inf

y_train_clf = pd.cut(y_train_reg, bins=bin_edges_open, labels=False, include_lowest=True)
y_test_clf = pd.cut(y_test_reg, bins=bin_edges_open, labels=False, include_lowest=True)

print("Faixas de geek_rating (definidas a partir do treino):")
for i, lbl in enumerate(bin_labels_desc):
    print(f"  Classe {i}: {lbl}")


In [ ]:
dist_treino = y_train_clf.value_counts().sort_index()
dist_teste = y_test_clf.value_counts().sort_index()

dist_df = pd.DataFrame({"treino": dist_treino, "teste": dist_teste}).fillna(0).astype(int)
print(dist_df)

plt.figure(figsize=(9, 4))
dist_df.plot(kind="bar", ax=plt.gca())
plt.title("Distribuição das classes (faixas de geek_rating)")
plt.xlabel("Classe (faixa)")
plt.ylabel("Quantidade de jogos")
plt.tight_layout()
plt.show()


⚠️ **Forte desbalanceamento de classes.** As classes 0 e 1 concentram a grande maioria dos jogos; as classes mais altas (7, 8, 9) têm poucas dezenas de exemplos. Por isso:

- Usamos `class_weight="balanced"` na Regressão Logística, para compensar o desbalanceamento durante o treino.
- Avaliamos com **F1-score ponderado (weighted)** como métrica principal do GridSearchCV (mais robusta a desbalanceamento que a acurácia simples), além de reportar accuracy e o relatório completo por classe.

## 6. Função auxiliar de avaliação (regressão)

Função utilitária para reportar RMSE, MAE e R² de forma consistente entre os 3 modelos de regressão.

In [ ]:
def avaliar_regressao(nome_modelo, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"--- {nome_modelo} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R²:   {r2:.4f}")

    return {"modelo": nome_modelo, "RMSE": rmse, "MAE": mae, "R2": r2}


## 7. Modelo 1 — Regressão Linear

Modelo baseline. O grid de hiperparâmetros é pequeno (a Regressão Linear tem poucos parâmetros relevantes), mas ainda assim passamos pelo `GridSearchCV` para manter a metodologia consistente entre os modelos.

In [ ]:
param_grid_linear = {
    "fit_intercept": [True, False],
    "positive": [True, False],
}

t0 = time.time()
grid_linear = GridSearchCV(
    estimator=LinearRegression(),
    param_grid=param_grid_linear,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_linear.fit(X_train, y_train_reg)

print(f"Tempo de treinamento: {time.time() - t0:.1f}s")
print(f"Melhores hiperparâmetros: {grid_linear.best_params_}")
print(f"Melhor R² (validação cruzada): {grid_linear.best_score_:.4f}")


In [ ]:
modelo_linear = grid_linear.best_estimator_
resultado_linear = avaliar_regressao("Regressão Linear", modelo_linear, X_test, y_test_reg)


## 8. Modelo 2 — Random Forest Regressor

Random Forest costuma capturar bem relações não-lineares entre as features (ex: número de votos, complexidade, categorias) e o `geek_rating`.

> 💡 **Nota de performance:** o grid abaixo foi dimensionado para terminar em alguns minutos em uma máquina comum. Se você tiver mais núcleos de CPU disponíveis, pode expandir as listas de `n_estimators`/`max_depth`/`min_samples_leaf` para uma busca mais ampla — o `n_jobs=-1` já paraleliza entre as combinações do grid. Tempo estimado com o grid abaixo: ~10-30 min em CPU com poucos núcleos; bem mais rápido com 4+ núcleos.

In [ ]:
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 4],
}

t0 = time.time()
grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=param_grid_rf,
    cv=3,
    scoring="r2",
    n_jobs=-1,
)
grid_rf.fit(X_train, y_train_reg)

print(f"Tempo de treinamento: {time.time() - t0:.1f}s")
print(f"Melhores hiperparâmetros: {grid_rf.best_params_}")
print(f"Melhor R² (validação cruzada): {grid_rf.best_score_:.4f}")


In [ ]:
modelo_rf = grid_rf.best_estimator_
resultado_rf = avaliar_regressao("Random Forest", modelo_rf, X_test, y_test_reg)


In [ ]:
# Importância das features (Random Forest)
importancias = pd.Series(modelo_rf.feature_importances_, index=X_train.columns)
top15 = importancias.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
top15.sort_values().plot(kind="barh")
plt.title("Top 15 features mais importantes — Random Forest")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()


## 9. Modelo 3 — Gradient Boosting Regressor

Gradient Boosting ajusta árvores sequencialmente, cada uma corrigindo os erros da anterior. Tende a ter desempenho competitivo, ao custo de mais tempo de treino.

> 💡 Mesma observação de performance do Random Forest: o grid foi dimensionado para tempo razoável (~5-15 min em CPU com poucos núcleos). Aumente se tiver capacidade computacional disponível.

In [ ]:
param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3],
}

t0 = time.time()
grid_gb = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid_gb,
    cv=3,
    scoring="r2",
    n_jobs=-1,
)
grid_gb.fit(X_train, y_train_reg)

print(f"Tempo de treinamento: {time.time() - t0:.1f}s")
print(f"Melhores hiperparâmetros: {grid_gb.best_params_}")
print(f"Melhor R² (validação cruzada): {grid_gb.best_score_:.4f}")


In [ ]:
modelo_gb = grid_gb.best_estimator_
resultado_gb = avaliar_regressao("Gradient Boosting", modelo_gb, X_test, y_test_reg)


## 10. Modelo 4 — Regressão Logística (classificação em 10 faixas)

Usamos `y_train_clf` / `y_test_clf` (as 10 faixas definidas na seção 5). `class_weight="balanced"` ajusta os pesos das classes para compensar o desbalanceamento.

> Nota: com 10 classes e forte desbalanceamento, é normal ver um aviso de convergência do solver `lbfgs` mesmo com `max_iter` razoavelmente alto — isso não impede o uso do modelo, apenas indica que o otimizador não atingiu o ponto de mínimo absoluto. Testes preliminares mostraram que aumentar `max_iter` muito além de 1000 não melhora o F1 no teste de forma perceptível, então mantivemos um valor que equilibra qualidade e tempo de treino.

In [ ]:
param_grid_logistic = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs"],
}

t0 = time.time()
grid_logistic = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    param_grid=param_grid_logistic,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
)
grid_logistic.fit(X_train, y_train_clf)

print(f"Tempo de treinamento: {time.time() - t0:.1f}s")
print(f"Melhores hiperparâmetros: {grid_logistic.best_params_}")
print(f"Melhor F1 ponderado (validação cruzada): {grid_logistic.best_score_:.4f}")


In [ ]:
modelo_logistic = grid_logistic.best_estimator_
y_pred_clf = modelo_logistic.predict(X_test)

acc = accuracy_score(y_test_clf, y_pred_clf)
f1_weighted = f1_score(y_test_clf, y_pred_clf, average="weighted")
precision_weighted = precision_score(y_test_clf, y_pred_clf, average="weighted", zero_division=0)
recall_weighted = recall_score(y_test_clf, y_pred_clf, average="weighted", zero_division=0)

print("--- Regressão Logística (10 classes) ---")
print(f"Accuracy:           {acc:.4f}")
print(f"F1 (weighted):       {f1_weighted:.4f}")
print(f"Precision (weighted): {precision_weighted:.4f}")
print(f"Recall (weighted):    {recall_weighted:.4f}")
print()
print(classification_report(y_test_clf, y_pred_clf, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test_clf, y_pred_clf, labels=range(N_BINS))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(N_BINS), yticklabels=range(N_BINS))
plt.title("Matriz de confusão — Regressão Logística (10 faixas)")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.tight_layout()
plt.show()


In [ ]:
resultado_logistic = {
    "modelo": "Regressão Logística (10 classes)",
    "Accuracy": acc,
    "F1_weighted": f1_weighted,
    "Precision_weighted": precision_weighted,
    "Recall_weighted": recall_weighted,
}


## 11. Comparação final dos modelos

Como a Regressão Logística resolve um problema diferente (classificação em 10 faixas, e não regressão contínua), reportamos suas métricas separadamente das métricas de regressão (RMSE, MAE, R²) dos outros 3 modelos.

In [ ]:
tabela_regressao = pd.DataFrame([resultado_linear, resultado_rf, resultado_gb]).set_index("modelo")
tabela_regressao = tabela_regressao.sort_values("R2", ascending=False)
print("Comparação — Modelos de Regressão (target contínuo)")
tabela_regressao


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metrica in zip(axes, ["RMSE", "MAE", "R2"]):
    tabela_regressao[metrica].plot(kind="bar", ax=ax, color="#4C72B0")
    ax.set_title(metrica)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
tabela_classificacao = pd.DataFrame([resultado_logistic]).set_index("modelo")
print("Resultado — Regressão Logística (target discretizado em 10 faixas)")
tabela_classificacao


## 12. Resumo dos melhores hiperparâmetros encontrados (GridSearchCV)

In [ ]:
resumo_hiperparametros = pd.DataFrame([
    {"modelo": "Regressão Linear", "melhores_hiperparametros": grid_linear.best_params_, "score_cv": grid_linear.best_score_, "metrica_cv": "R2"},
    {"modelo": "Random Forest", "melhores_hiperparametros": grid_rf.best_params_, "score_cv": grid_rf.best_score_, "metrica_cv": "R2"},
    {"modelo": "Gradient Boosting", "melhores_hiperparametros": grid_gb.best_params_, "score_cv": grid_gb.best_score_, "metrica_cv": "R2"},
    {"modelo": "Regressão Logística", "melhores_hiperparametros": grid_logistic.best_params_, "score_cv": grid_logistic.best_score_, "metrica_cv": "F1_weighted"},
])
resumo_hiperparametros


## 13. Conclusões

- Os 3 modelos de regressão (Linear, Random Forest, Gradient Boosting) são diretamente comparáveis por RMSE, MAE e R², pois preveem o `geek_rating` na escala original.
- A Regressão Logística resolve uma tarefa diferente (classificar em qual de 10 faixas o jogo cai) e é avaliada por métricas de classificação (accuracy, F1, precision, recall).
- O forte desbalanceamento de classes em `geek_rating` (poucos jogos com rating muito alto) afeta principalmente a Regressão Logística nas faixas superiores — isso é esperado e reflete a própria distribuição do Bayesian rating do BGG, não um erro no pipeline.
- O `GridSearchCV` identificou os melhores hiperparâmetros para cada modelo dentro dos grids definidos. Se desejar uma busca mais ampla (mais valores por hiperparâmetro), os grids podem ser expandidos — o principal custo adicional cairá sobre Random Forest e Gradient Boosting, que são os mais caros computacionalmente.